# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. `1 1.3333333333333333`. Пропуск один. После замены на ноль среднее равно `(1 + 0 + 3) / 3`.
2. `2`. После удаления пробелов и перевода в нижний регистр получаются `a`, `a`, `b`.
3. `3`. `drop_duplicates()` возвращает новую таблицу. Результат не присвоили, поэтому в `df` остались три строки.
4. `1005`. После удаления пробела и перевода в числа складываются `1000` и `5`.
5. `[9, 10]`. `.dt.month` берёт номер месяца из каждой даты.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
# 1
import pandas as pd

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print("После загрузки:", len(df))

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("Пропусков в amount:", df["amount"].isna().sum())
    print("После перевода сумм в числа:", len(df))

    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("Пропусков в city:", df["city"].isna().sum())
    print("После приведения названий городов:", len(df))

    df = df.drop_duplicates().copy()
    print("Все order_id уникальны:", df["order_id"].is_unique)
    print("После удаления дублей:", len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print("После перевода дат:", len(df))

    df = df[df["customer_id"] != "TEST"]
    print("После исключения TEST:", len(df))
    return df

orders = clean("orders.csv")
print(f"Для отчёта осталось {len(orders)} заказов. Отменённые оставили для расчёта доли отмен.")

После загрузки: 5463
Пропусков в amount: 0
После перевода сумм в числа: 5463
Пропусков в city: 0
После приведения названий городов: 5463
Все order_id уникальны: True
После удаления дублей: 5283
После перевода дат: 5283
После исключения TEST: 5280
Для отчёта осталось 5280 заказов. Отменённые оставили для расчёта доли отмен.


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
# 2
paid = orders["status"] == "оплачен"
sep = orders["month"] == 9
octo = orders["month"] == 10

sep_revenue = orders.loc[paid & sep, "amount"].sum()
oct_revenue = orders.loc[paid & octo, "amount"].sum()
change = oct_revenue / sep_revenue - 1
print(f"Всего: сентябрь {sep_revenue:,.0f} ₸, октябрь {oct_revenue:,.0f} ₸, изменение {change:+.1%}")

city_changes = {}
for city in orders["city"].unique():
    city_orders = orders["city"] == city
    s = orders.loc[paid & city_orders & sep, "amount"].sum()
    o = orders.loc[paid & city_orders & octo, "amount"].sum()
    city_changes[city] = o / s - 1
    print(f"{city}: сентябрь {s:,.0f} ₸, октябрь {o:,.0f} ₸, изменение {city_changes[city]:+.1%}")

changes = pd.Series(city_changes).sort_values()
worst_city = changes.index[0]
print("Суммы по городам совпадают с таблицей урока 4.")
print(f"Начните проверку с города {worst_city}. Там выручка упала сильнее всего ({changes.iloc[0]:+.1%}).")

Всего: сентябрь 35,243,410 ₸, октябрь 34,419,520 ₸, изменение -2.3%
Алматы: сентябрь 14,501,070 ₸, октябрь 14,643,020 ₸, изменение +1.0%
Астана: сентябрь 9,034,220 ₸, октябрь 8,850,530 ₸, изменение -2.0%
Караганда: сентябрь 3,648,520 ₸, октябрь 3,371,680 ₸, изменение -7.6%
Шымкент: сентябрь 4,558,650 ₸, октябрь 4,044,010 ₸, изменение -11.3%
Атырау: сентябрь 923,990 ₸, октябрь 969,860 ₸, изменение +5.0%
Актобе: сентябрь 2,576,960 ₸, октябрь 2,540,420 ₸, изменение -1.4%
Суммы по городам совпадают с таблицей урока 4.
Начните проверку с города Шымкент. Там выручка упала сильнее всего (-11.3%).


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

| Шаг очистки | Строк убрали | Строк осталось | Изменение выручки в октябре к сентябрю |
|---|---:|---:|---:|
| Исходные данные, выручка только по оплаченным | 0 | 5 463 | −22,7% |
| Перевели суммы в числа | 0 | 5 463 | −22,7% |
| Привели названия городов | 0 | 5 463 | −22,7% |
| Убрали дубли | 180 | 5 283 | −19,6% |
| Перевели даты | 0 | 5 283 | −19,6% |
| Исключили TEST | 3 | 5 280 | −2,3% |

Сильнее всего оценку падения исказили заказы TEST. В сентябре их сумма была примерно на 10 млн ₸ выше, чем в октябре, поэтому исключите их из отчёта.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [4]:
# 4
missing = orders["delivery_days"].isna()
print("Всего пропусков:", missing.sum())
print("По городам:")
print(orders.loc[missing, "city"].value_counts())
print("По статусам:")
print(orders.loc[missing, "status"].value_counts())

days = orders.loc[orders["city"] == "Шымкент", "delivery_days"]
known_days = days.dropna()
median_days = days.median()
drop_share = (known_days <= 2).mean()
median_share = (days.fillna(median_days) <= 2).mean()
zero_share = (days.fillna(0) <= 2).mean()

print(f"За 2 дня или быстрее в Шымкенте: dropna — {drop_share:.1%}, "
      f"fillna({median_days:g}) — {median_share:.1%}, fillna(0) — {zero_share:.1%}.")
print(f"В отчёте покажите {drop_share:.1%} среди {len(known_days)} заказов с известным сроком. "
      f"Подстановка медианы {median_days:g} дней считает пропуски медленными доставками, "
      "а подстановка нуля считает их мгновенными и завышает долю.")

Всего пропусков: 158
По городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
По статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64
За 2 дня или быстрее в Шымкенте: dropna — 29.8%, fillna(5) — 29.0%, fillna(0) — 31.8%.
В отчёте покажите 29.8% среди 754 заказов с известным сроком. Подстановка медианы 5 дней считает пропуски медленными доставками, а подстановка нуля считает их мгновенными и завышает долю.


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [5]:
# 5
# drop_duplicates() возвращает новую таблицу. Сохраняем результат.
# lower() исправляет только регистр. Нужны ещё strip() и словарь CITY.
df = pd.read_csv("orders.csv")
df = df.drop_duplicates().copy()
df["city"] = df["city"].str.strip().str.lower().map(CITY)

print(f"После исправления {df['city'].nunique()} городов и {len(df)} строк.")
print("Для отчёта объединяйте варианты написания города и учитывайте каждый заказ один раз.")

После исправления 6 городов и 5283 строк.
Для отчёта объединяйте варианты написания города и учитывайте каждый заказ один раз.


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [6]:
# 6
weekday_names = {"Monday": "Понедельник", "Tuesday": "Вторник", "Wednesday": "Среда",
                 "Thursday": "Четверг", "Friday": "Пятница", "Saturday": "Суббота",
                 "Sunday": "Воскресенье"}
weekday_counts = orders["order_date"].dt.day_name().map(weekday_names).value_counts()
print(weekday_counts)

busiest_day = weekday_counts.index[0]
quietest_day = weekday_counts.index[-1]
busiest_count = weekday_counts.iloc[0]
quietest_count = weekday_counts.iloc[-1]
gap = (busiest_count - quietest_count) / busiest_count

print(f"{quietest_day}: {quietest_count} заказов, на {gap:.1%} меньше самого загруженного дня "
      f"({busiest_day.lower()}, {busiest_count} заказов).")
print("Запланируйте больше курьеров на четверг. При составлении смен учитывайте также число доставок по часам.")

order_date
Четверг        786
Вторник        785
Пятница        781
Среда          774
Суббота        773
Воскресенье    697
Понедельник    684
Name: count, dtype: int64
Понедельник: 684 заказов, на 13.0% меньше самого загруженного дня (четверг, 786 заказов).
Запланируйте больше курьеров на четверг. При составлении смен учитывайте также число доставок по часам.


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [7]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [8]:
# 8